# 第十一章 函数进阶：内置函数、lambda 与组合


## 初学者学习路线

这章建议按照“先观察、再模仿、后修改、最后独立完成”的顺序学习，不必一次记住所有参数。

1. 先阅读任务说明，明确这段代码要回答什么问题。
2. 运行一个最小例子，先观察输入、输出和数据形状，再回看每一行代码。
3. 只修改一个参数或一条数据，重新运行并比较前后结果。
4. 完成“综合练习”，最后再看本章小结，把能迁移到其他数据的问题写下来。

运行时如果看到 NameError，通常是前置单元格还没有运行；如果输出和预期不同，先检查变量是否被后面的单元格重新赋值。


## 先做一个小检查

进入正式例子前，先用一句话回答：本章的输入是什么，想得到什么结果？

本章主题是“第十一章 函数进阶：内置函数、lambda 与组合”。请特别留意三件事：输入的类型或形状、处理中间变量的含义、最后输出能否支持一个清楚的结论。


## 本章场景

记账程序的分析需求越来越复杂：找出所有支出并按金额排序、统计最大支出、按条件筛选记录。这些任务有一个共同模式：**把“规则”作为参数传给通用函数**。

本章学习 Python 的函数式工具：**key 参数**（排序/最值依据）、**lambda**（临时函数）、**filter/map**（批量处理）、***args/**kwargs**（参数收集与展开）。学完后，你将能写出通用、简洁、可复用的账目分析代码。

1. 11.1 key 参数与函数对象
2. 11.2 推导式、filter 与 map
3. 11.3 *args 与 **kwargs
4. 11.4 工具速查
5. 11.5 本章实训：组合分析账目
6. 11.6 易错点提醒
7. 11.7 练习与作业
8. 11.8 小结
9. 11.9 拓展作业



## Python 的学习主线

概念与语法 → 最小可运行代码 → 修改一个输入 → 处理边界条件 → 封装成函数 → 独立完成一个小任务

先预测输出，再运行代码；然后只改一个变量，最后把示例改写成自己的问题。重点检查变量类型、条件分支和中间结果。


## 本模块练习方式

基础：补全或改写一小段代码；提高：组合两个语法知识点；挑战：处理空输入、错误类型或边界值。

完成后请写下：输入是什么、处理做了什么、输出说明了什么、还存在什么限制。


## 本章目标

学完本章，你将能够：

- **理解**：理解内置函数、lambda、可迭代解包与函数组合。
- **操作**：能用 lambda + 内置函数（sorted/map/filter）简化数据处理。
- **迁移**：能把多个小函数组合成一条数据处理流水线。


## 11.1 key 参数与函数对象

**背景引入**：上一章学会了 def 定义函数，但函数的价值不止于“自己调用”——函数还可以作为参数传给其他函数。sorted 的 key 参数就是最典型的例子：告诉排序“按哪个字段排”。这是函数进阶的第一课。（把排序规则想成一张“称重卡”：交给 sorted，它就对每个元素照这张卡取数值来排。）

### 11.1.1 函数是一等对象

函数可以像变量一样传递：sorted(records, key=amount_of) 中的 key 接收一个**函数**，排序时对每个元素调用它取排序依据。

### 11.1.2 命名函数与 lambda

- 命名函数：def amount_of(record): return record["amount"]；
- lambda：lambda record: record["amount"]——匿名、单表达式、即写即用。

两者等价；复杂逻辑用命名函数，简单取字段用 lambda。




**练一练 11.1**：把三条账目按金额从大到小排序（不修改原列表），并找出金额最大的那条记录。


In [ ]:
# 请在下方填写代码
records = [
    {"category": "餐饮", "amount": 35.5},
    {"category": "购物", "amount": 299.0},
    {"category": "交通", "amount": 18.0},
]
# TODO：请在下方完成 —— 练一练 11.1：把三条账目按金额从大到小排序（不修改原列表），并找出金额最大的那条记录。


In [ ]:
records = [
    {"category": "餐饮", "amount": 35.5},
    {"category": "购物", "amount": 299.0},
    {"category": "交通", "amount": 18.0},
]
sorted_records = sorted(records, key=lambda item: item["amount"], reverse=True)
print(sorted_records)
print(max(records, key=lambda item: item["amount"]))


In [ ]:
records = [
    {"category": "餐饮", "amount": 35.5},
    {"category": "购物", "amount": 299.0},
    {"category": "交通", "amount": 18.0},
]


def amount_of(record):
    return record["amount"]


ranked = sorted(records, key=amount_of, reverse=True)
largest = max(records, key=lambda record: record["amount"])
print(ranked)
print("最大一笔：", largest)


**输出解读**：sorted 按金额降序排列；max 用 key 找金额最大的记录——函数作为参数实现了“按什么排序/取最值”的定制。


## 11.2 推导式、filter 与 map

**背景引入**：筛选“支出类账目”、把金额统一转成字符串——这类“批量变换”除了列表推导式，还有 filter（筛选）和 map（变换）两种函数式写法。三者各有所长，本节讲清怎么选。

filter(规则函数, 容器) 筛选；map(变换函数, 容器) 变换。两者返回**惰性迭代器**，需 list() 消费。与推导式等价：

- filter + map：list(map(lambda r: r["amount"], filter(lambda r: r["type"] == "支出", records)))；
- 推导式：[r["amount"] for r in records if r["type"] == "支出"]——**更推荐推导式**。




**练一练 11.2**：从账目列表中筛选出支出记录（filter），再把金额转成字符串（map），最后把两个结果转成列表打印。


In [ ]:
# 请在下方填写代码
records = [
    {"type": "支出", "amount": 35.5},
    {"type": "收入", "amount": 5000.0},
    {"type": "支出", "amount": 299.0},
]
# TODO：请在下方完成 —— 练一练 11.2：从账目列表中筛选出支出记录（filter），再把金额转成字符串（map），最后把两个结果转成列表打印。


In [ ]:
records = [
    {"type": "支出", "amount": 35.5},
    {"type": "收入", "amount": 5000.0},
    {"type": "支出", "amount": 299.0},
]
expenses = list(filter(lambda r: r["type"] == "支出", records))
amount_texts = list(map(lambda r: str(r["amount"]), expenses))
print(expenses)
print(amount_texts)


In [ ]:
records = [
    {"type": "支出", "amount": 35.5},
    {"type": "收入", "amount": 5000.0},
    {"type": "支出", "amount": 18.0},
]
expenses = list(filter(lambda record: record["type"] == "支出", records))
amounts = list(map(lambda record: record["amount"], expenses))
same_amounts = [
    record["amount"] for record in records if record["type"] == "支出"
]
print(expenses, amounts, same_amounts, sum(amounts))


**输出解读**：filter 筛出支出、map 取出金额；推导式得到相同结果——可读性上推导式更胜一筹。


## 11.3 *args 与 **kwargs

**背景引入**：有时调用方传多少参数事先不确定——比如“任意多笔金额求合计”。*args 收集任意多个位置参数为元组，**kwargs 收集任意多个关键字参数为字典，让函数接口更灵活。

### 11.3.1 参数收集

- *args 收集所有多余位置参数为元组：def total_amount(*amounts)；
- **kwargs 收集所有多余关键字参数为字典：def create_record(date, record_type, amount, **extras)。

### 11.3.2 参数展开（调用时）

- f(*列表) 把列表展开为位置参数；
- f(**字典) 把字典展开为关键字参数。




**练一练 11.3**：定义 total_amount(*amounts) 对任意多笔金额求和并打印；再定义 build_record(**fields) 把任意键值打包成字典。


In [ ]:
# 请在下方填写代码
# TODO：请在下方完成 —— 练一练 11.3：定义 total_amount(amounts) 对任意多笔金额求和并打印；再定义 build_rec


In [ ]:
def total_amount(*amounts):
    return sum(amounts)


print(total_amount(35.5, 18.0, 299.0))


def build_record(**fields):
    return fields


record = build_record(date="2026-08-06", category="餐饮", amount=35.5)
print(record)


In [ ]:
def total_amount(*amounts):
    return sum(amounts)


def create_record(date, record_type, amount, **extras):
    record = {"date": date, "type": record_type, "amount": amount}
    record.update(extras)
    return record


print(total_amount(35.5, 18.0, 299.0))
print(create_record("2026-08-06", "支出", 35.5, category="餐饮", note="午餐"))


**输出解读**：*amounts 收集三个金额求和；**extras 收集 category 与 note 合并进记录字典。


## 11.4 工具速查

| 函数 | 用途 | 记账场景 |
| --- | --- | --- |
| len(容器) | 长度 | 账目笔数 |
| sum(可迭代) | 求和 | 金额合计 |
| min / max(可迭代, key=, default=) | 最小 / 最大 | 最大支出（default 防空） |
| abs(数值) | 绝对值 | 差额 |
| round(数值, 位数) | 舍入 | 金额显示 |
| sorted(可迭代, key=, reverse=) | 排序 | 按日期/金额排序 |
| zip / enumerate | 配对 / 序号 | 字段配对 |
| any / all | 任一 / 全部 | 规则检查 |
| dict(zip(键, 值)) | 构造字典 | 字段构造 |




## 11.5 本章实训：组合分析账目

本节 3 个实验覆盖筛选、排序与统计的组合。




### 实验 1：按字段排序与取最值

**操作步骤**：运行下方代码；把 reverse=True 去掉观察顺序变化。

**观察要点**：key 函数决定排序依据；lambda 即写即用。


In [ ]:
# 实验 1：key 参数
records = [
    {"category": "餐饮", "amount": 35.5},
    {"category": "购物", "amount": 299.0},
    {"category": "交通", "amount": 18.0},
]
ranked = sorted(records, key=lambda record: record["amount"], reverse=True)
largest = max(records, key=lambda record: record["amount"])
print(ranked)
print("最大一笔：", largest)


**结果记录**：排序结果与最大记录是什么？


### 实验 2：筛选 + 变换

**操作步骤**：运行下方代码；比较 filter/map 与推导式的等价性。

**观察要点**：filter/map 惰性，必须 list() 消费。


In [ ]:
# 实验 2：filter + map
records = [
    {"type": "支出", "amount": 35.5},
    {"type": "收入", "amount": 5000.0},
    {"type": "支出", "amount": 18.0},
]
expenses = list(filter(lambda record: record["type"] == "支出", records))
amounts = list(map(lambda record: record["amount"], expenses))
same = [record["amount"] for record in records if record["type"] == "支出"]
print(expenses, amounts, same, sum(amounts))


**结果记录**：amounts 与 same 是否相同？sum 的结果是什么？


### 实验 3：多条件查询函数

**操作步骤**：运行下方代码；用 category="餐饮" 再调用一次。

**观察要点**：可选参数过滤 + sorted 排序组合成查询函数。


In [ ]:
# 实验 3：多条件查询
def search_records(records, record_type=None, category=None):
    result = records
    if record_type is not None:
        result = [item for item in result if item["type"] == record_type]
    if category is not None:
        result = [item for item in result if item["category"] == category]
    return sorted(result, key=lambda item: item["date"])


records = [
    {"date": "2026-08-07", "type": "支出", "category": "交通", "amount": 18.0},
    {"date": "2026-08-06", "type": "支出", "category": "餐饮", "amount": 35.5},
]
print(search_records(records, record_type="支出"))


**结果记录**：按支出筛选后的结果是什么？


## 11.6 易错点提醒




### 11.6.1 len、sum、min、max、abs 和 round

min/max 的 default 参数防空列表：max([], default=None) 返回 None 而不是报错。round 与格式化区别：round 返回数值，:.2f 返回字符串。




In [ ]:
amounts = [35.5, 18.0, 299.0]
print("笔数：", len(amounts))
print("合计：", sum(amounts))
print("最小：", min(amounts, default=None))
print("最大：", max(amounts, default=None))
print("差额绝对值：", abs(35.5 - 53.5))
print("平均值：", round(sum(amounts) / len(amounts), 2))
print("空列表最大值：", max([], default=None))


**输出解读**：统计函数一行完成账目指标；default 参数让空列表安全返回 None。


### 11.6.2 enumerate、zip、any 和 all 组合使用

dict(zip(字段, 值)) 构造记录；enumerate 加序号；any/all 汇总检查。




In [ ]:
fields = ["date", "type", "category", "amount"]
values = ["2026-08-06", "支出", "餐饮", 35.5]
record = dict(zip(fields, values))
for number, (key, value) in enumerate(record.items(), start=1):
    print(number, key, value)

required = [record.get(field) for field in fields]
print("必填都存在：", all(required))
print("包含大额：", any(value > 5000 for value in [35.5, 18.0]))


**输出解读**：zip 配对字段与值构造字典；all 检查必填；any 生成器表达式逐值判断。


### 11.6.3 把函数当参数：写一个通用筛选器

def select(records, predicate): 接收一个**规则函数**，返回满足规则的记录。筛选逻辑与筛选规则分离——规则可替换、可复用。




In [ ]:
def select(records, predicate):
    return [record for record in records if predicate(record)]


def is_expense(record):
    return record["type"] == "支出"


def is_large(record):
    return record["amount"] >= 5000


records = [
    {"type": "支出", "amount": 35.5},
    {"type": "收入", "amount": 5000.0},
]
print(select(records, is_expense))
print(select(records, is_large))
print(callable(is_expense))


**输出解读**：同一个 select 函数配合不同规则函数完成不同筛选；callable 验证函数对象。


### 11.6.4 * 和 **：调用时拆开参数

f(*元组) 按位置展开；f(**字典) 按关键字展开。与定义时的收集（*args/**kwargs）互为逆操作。




In [ ]:
def format_amount(category, amount, note=""):
    suffix = f"（{note}）" if note else ""
    return f"{category}：{amount:.2f} 元{suffix}"


positional = ("餐饮", 35.5)
keyword_data = {"category": "交通", "amount": 18.0, "note": "地铁"}
print(format_amount(*positional))
print(format_amount(**keyword_data))


**输出解读**：* 展开元组为位置参数；** 展开字典为关键字参数。


### 11.6.5 itemgetter：简单取字段不一定要 lambda

operator.itemgetter("amount") 返回一个取字段函数，可作 key 使用——比 lambda 更简短。




In [ ]:
from operator import itemgetter

records = [
    {"category": "餐饮", "amount": 35.5},
    {"category": "购物", "amount": 299.0},
    {"category": "交通", "amount": 18.0},
]
amount_of = itemgetter("amount")
print(sorted(records, key=amount_of))
print(max(records, key=amount_of))
print(list(map(amount_of, records)))


**输出解读**：itemgetter 生成的函数用于排序、取最值与批量取值，语义清晰。


### 11.6.6 map/filter 是惰性的，只能一路消费

filter/map 返回惰性迭代器：第一次 list() 消费后，再次转列表为空。需要多次使用时先转列表保存。




In [ ]:
records = [
    {"type": "支出", "amount": 35.5},
    {"type": "收入", "amount": 5000.0},
    {"type": "支出", "amount": 18.0},
]
expense_iterator = filter(lambda record: record["type"] == "支出", records)
first_pass = list(expense_iterator)
second_pass = list(expense_iterator)
amounts = [record["amount"] for record in first_pass]
print(first_pass)
print(second_pass)
print(amounts)


**输出解读**：第二次 list() 得到空列表——迭代器一次性；因此先用 first_pass 保存结果再复用。


## 11.7 练习与作业

练习分为基础、提高、挑战三级。先独立完成，再对照参考答案。




### 基础 1：key 排序

对 records（餐饮 35.5、购物 299.0、交通 18.0）按金额降序排列并输出。


In [ ]:
# 请在下方填写代码
records = [
    {"category": "餐饮", "amount": 35.5},
    {"category": "购物", "amount": 299.0},
    {"category": "交通", "amount": 18.0},
]
# TODO：请在下方完成 —— 基础 1：key 排序 对 records（餐饮 35.5、购物 299.0、交通 18.0）按金额降序排列并输出。


In [ ]:
records = [
    {"category": "餐饮", "amount": 35.5},
    {"category": "购物", "amount": 299.0},
    {"category": "交通", "amount": 18.0},
]
ranked = sorted(records, key=lambda record: record["amount"], reverse=True)
print(ranked)


### 基础 2：max 取最大支出

用 max + lambda 找出支出金额最大的记录（含 default=None 防空）。


In [ ]:
# 请在下方填写代码
records = [
    {"category": "餐饮", "amount": 35.5},
    {"category": "购物", "amount": 299.0},
]
# TODO：请在下方完成 —— 基础 2：max 取最大支出 用 max + lambda 找出支出金额最大的记录（含 default=None 防空）


In [ ]:
records = [
    {"category": "餐饮", "amount": 35.5},
    {"category": "购物", "amount": 299.0},
]
largest = max(records, key=lambda item: item["amount"], default=None)
print(largest)


### 基础 3：*args 求和

定义 total_amount(*amounts) 返回合计，用三个金额验证。


In [ ]:
# 请在下方填写代码
# TODO：请在下方完成 —— 基础 3：args 求和 定义 total_amount(amounts) 返回合计，用三个金额验证。


In [ ]:
def total_amount(*amounts):
    return sum(amounts)


print(total_amount(35.5, 18.0, 299.0))


### 提高 1：筛选 + 排序 + 最大支出

对三笔记录（含收入）筛选支出，按金额降序排列，并找出最大支出记录。


In [ ]:
# 请在下方填写代码
records = [
    {"type": "支出", "category": "餐饮", "amount": 35.5},
    {"type": "收入", "category": "工资", "amount": 5000.0},
    {"type": "支出", "category": "购物", "amount": 299.0},
]
# TODO：请在下方完成 —— 提高 1：筛选 + 排序 + 最大支出 对三笔记录（含收入）筛选支出，按金额降序排列，并找出最大支出记录。


In [ ]:
records = [
    {"type": "支出", "category": "餐饮", "amount": 35.5},
    {"type": "收入", "category": "工资", "amount": 5000.0},
    {"type": "支出", "category": "购物", "amount": 299.0},
]
expenses = list(filter(lambda item: item["type"] == "支出", records))
ranked = sorted(expenses, key=lambda item: item["amount"], reverse=True)
largest = max(expenses, key=lambda item: item["amount"], default=None)
print("排序后：", ranked)
print("最大支出：", largest)
_ok = largest["category"] == "购物"
_ok = _ok and records[0]["category"] == "餐饮"
print("诊断：", "通过" if _ok else "检查筛选或 key 函数")


### 提高 2：通用查询函数

定义 largest_record(records, record_type=None)：可选按类型筛选后返回金额最大记录（max + key + default）。


In [ ]:
# 请在下方填写代码
records = [
    {"type": "支出", "amount": 35.5},
    {"type": "收入", "amount": 5000.0},
]
# TODO：请在下方完成 —— 提高 2：通用查询函数 定义 largest_record(records, record_type=None)：可选按


In [ ]:
def largest_record(records, record_type=None):
    if record_type is not None:
        records = [item for item in records if item["type"] == record_type]
    return max(records, key=lambda item: item["amount"], default=None)


records = [
    {"type": "支出", "amount": 35.5},
    {"type": "收入", "amount": 5000.0},
]
print(largest_record(records, record_type="支出"))
print(largest_record(records))
_ok = largest_record(records, record_type="支出")["amount"] == 35.5
_ok = _ok and largest_record(records)["amount"] == 5000.0
print("诊断：", "通过" if _ok else "检查筛选或 key")


### 挑战 1：查询 + 统计函数组合

实现 search_records（按类型/分类过滤 + 按日期排序）与 statistics（收入合计、支出合计、结余、最大支出），并组合验证。


In [ ]:
# 请在下方填写代码
records = [
    {
        "date": "2026-08-01",
        "type": "收入",
        "category": "工资",
        "amount": 5000.0,
    },
    {"date": "2026-08-02", "type": "支出", "category": "餐饮", "amount": 35.5},
    {
        "date": "2026-08-03",
        "type": "支出",
        "category": "购物",
        "amount": 299.0,
    },
]
# TODO：请在下方完成 —— 挑战 1：查询 + 统计函数组合 实现 search_records（按类型/分类过滤 + 按日期排序）与 statis


In [ ]:
def search_records(records, record_type=None, category=None):
    result = records
    if record_type is not None:
        result = list(filter(lambda item: item["type"] == record_type, result))
    if category is not None:
        result = list(
            filter(lambda item: item["category"] == category, result)
        )
    return sorted(result, key=lambda item: item["date"])


def statistics(records):
    income = sum(item["amount"] for item in records if item["type"] == "收入")
    expenses = [item for item in records if item["type"] == "支出"]
    expense_total = sum(item["amount"] for item in expenses)
    largest_expense = max(
        expenses, key=lambda item: item["amount"], default=None
    )
    return {
        "income": income,
        "expense": expense_total,
        "balance": income - expense_total,
        "largest_expense": largest_expense,
    }


records = [
    {
        "date": "2026-08-01",
        "type": "收入",
        "category": "工资",
        "amount": 5000.0,
    },
    {"date": "2026-08-02", "type": "支出", "category": "餐饮", "amount": 35.5},
    {
        "date": "2026-08-03",
        "type": "支出",
        "category": "购物",
        "amount": 299.0,
    },
]
result = statistics(records)
print(search_records(records, record_type="支出"))
print(result)
_ok = (
    result["balance"] == 4665.5
    and result["largest_expense"]["amount"] == 299.0
)
print("诊断：", "通过" if _ok else "检查筛选、合计或 key")


## 11.8 小结

### 知识要点回顾

| 知识点 | 要点 |
| --- | --- |
| 函数对象 | 函数可作参数、可赋值 |
| key 参数 | sorted / max / min 的排序依据 |
| lambda | 匿名单表达式函数 |
| filter / map | 筛选 / 变换；惰性，list() 消费 |
| *args / **kwargs | 收集多余参数 |
| * / ** 展开 | 调用时拆开元组 / 字典 |
| itemgetter | 取字段函数，替代简单 lambda |
| 统计函数 | len / sum / min / max / abs / round |

### 自测清单

- [ ] 能用 key + lambda 按字段排序与取最值；
- [ ] 能写出通用筛选器（函数作参数）；
- [ ] 能解释 filter/map 的惰性；
- [ ] 能用 *args/**kwargs 收集与展开参数；
- [ ] 能用 itemgetter 简化取字段；
- [ ] 能用统计函数完成账目指标计算。




## 11.9 拓展作业

### 必做作业

**作业 1（组合分析）**：给定 5 笔记录，用 filter + sorted + max 完成：筛出支出、按金额降序、找出最大支出与支出合计。

**作业 2（统计函数）**：编写 statistics(records) 返回 {income, expense, balance, largest_expense}，并用 3 笔记录验证。

### 选做拓展

研究 functools.reduce 与 sum 的关系，用 reduce 实现金额合计，并讨论可读性。

### 下章预习

第 12 章《文件、路径与 JSON 持久化》将学习：pathlib 路径操作、with 读写文件、JSON 序列化。预习时思考：账目数据如何保存到磁盘并在下次启动时恢复？




## 教学实验：边界条件

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
orders = [
    {"order_id": "A01", "amount": 280},
    {"order_id": "A02", "amount": 300},
    {"order_id": "A03", "amount": 520},
]
threshold = 300
for order in orders:
    label = "达到门槛" if order["amount"] >= threshold else "未达到门槛"
    print(order["order_id"], order["amount"], label)


### 第一个结果怎么读

这里的重点不是记住 `if`，而是观察 `>=` 如何处理刚好等于 300 的记录。边界条件必须和业务规则保持一致。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
threshold = 500
for order in orders:
    label = "重点关注" if order["amount"] >= threshold else "普通订单"
    print(order["order_id"], "->", label)
print("重点订单数：", sum(order["amount"] >= threshold for order in orders))


### 第二个结果怎么读

只把门槛从 300 改成 500，再比较标签和数量变化。这个实验训练的是“改一个输入，解释一个输出”。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：类型转换失败怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
amount_text = "128.5"
try:
    amount = int(amount_text)
except ValueError as error:
    print("第一次转换失败：", type(error).__name__)
    amount = float(amount_text)
print("可以继续使用的金额：", amount)


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

先读错误类型，再决定修复方法。这里不是盲目忽略错误，而是明确知道整数转换不适合带小数的文本。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。
